# Phase 1 — First look at TCGA-BRCA

**Author:** [Your name]
**Course:** Statistics — Fall 1405
**Date:** 1405-XX-XX
**Path:** ب (TCGA-BRCA multi-omic)

## Goal of this notebook
1. Audit the raw structure of the TCGA-BRCA clinical table.
2. Diagnose and document its row duplication.
3. Collapse it to a clean **one-row-per-patient** table.
4. (Later) Map RNA-seq files to patients, merge, and produce
   first descriptive plots.

## Data
- Clinical: `data/raw/clinical/clinical.tsv` (GDC export, 2026-10-31)
- RNA-seq : `data/raw/rnaseq/<file_id>/<file_id>.rna_seq...tsv` (1,231 files)# Phase 1 — First look at TCGA-BRCA

**Author:** [Your name]
**Course:** Statistics — Fall 1405
**Date:** 1405-XX-XX
**Path:** ب (TCGA-BRCA multi-omic)

## Goal of this notebook
1. Audit the raw structure of the TCGA-BRCA clinical table.
2. Diagnose and document its row duplication.
3. Collapse it to a clean **one-row-per-patient** table.
4. (Later) Map RNA-seq files to patients, merge, and produce
   first descriptive plots.

## Data
- Clinical: `data/raw/clinical/clinical.tsv` (GDC export, 2026-10-31)
- RNA-seq : `data/raw/rnaseq/<file_id>/<file_id>.rna_seq...tsv` (1,231 files)

## 1. Setup and raw load

In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CLINICAL_PATH = "../data/raw/clinical/clinical.tsv"
RNASEQ_DIR    = "../data/raw/rnaseq"

assert os.path.exists(CLINICAL_PATH), f"File not found: {os.path.abspath(CLINICAL_PATH)}"
assert os.path.isdir(RNASEQ_DIR),     f"Dir not found:  {os.path.abspath(RNASEQ_DIR)}"

# RNA-seq files (verified earlier: 1,231)
rnaseq_files = glob.glob(os.path.join(RNASEQ_DIR, "*", "*.tsv"))
print("RNA-seq .tsv files found:", len(rnaseq_files))

# Raw clinical table
df = pd.read_csv(CLINICAL_PATH, sep="\t", low_memory=False)
print("Raw clinical shape:", df.shape)

RNA-seq .tsv files found: 1231
Raw clinical shape: (5546, 202)


### Column audit
The clinical table is node-prefixed: `cases.*`, `demographic.*`,
`diagnoses.*`, `treatments.*`. The patient key is `cases.case_id`.
GDC stores missing values as the literal string `"'--"` (not `NaN`).
Boolean flags are lowercase strings (`"true"` / `"false"`).

In [2]:
print("All columns (%d):" % df.shape[1])
for c in df.columns:
    print(" ", c)

All columns (202):
  project.project_id
  cases.case_id
  cases.consent_type
  cases.days_to_consent
  cases.days_to_lost_to_followup
  cases.disease_type
  cases.index_date
  cases.lost_to_followup
  cases.primary_site
  cases.submitter_id
  demographic.age_at_index
  demographic.age_is_obfuscated
  demographic.cause_of_death
  demographic.cause_of_death_source
  demographic.country_of_birth
  demographic.country_of_residence_at_enrollment
  demographic.days_to_birth
  demographic.days_to_death
  demographic.demographic_id
  demographic.education_level
  demographic.ethnicity
  demographic.marital_status
  demographic.occupation_duration_years
  demographic.population_group
  demographic.race
  demographic.sex_at_birth
  demographic.submitter_id
  demographic.vital_status
  demographic.year_of_birth
  demographic.year_of_birth_range
  demographic.year_of_death
  diagnoses.adrenal_hormone
  diagnoses.age_at_diagnosis
  diagnoses.ajcc_clinical_m
  diagnoses.ajcc_clinical_n
  diagnoses.a

## 2. Diagnose the row duplication

The raw table has **5,546 rows** but only **1,098 unique patients**.
The goal of this section is to prove that the duplication is
**structural** (patient × diagnosis × treatment), not a data error.

In [3]:
print("Total rows:              ", len(df))
print("Unique case_id:          ", df["cases.case_id"].nunique())
print("Unique submitter_id:     ", df["cases.submitter_id"].nunique())

print("\nDistribution of rows per case:")
print(df.groupby("cases.case_id").size().value_counts().sort_index())

print("\nUnique (case, diagnosis) pairs:",
      df.groupby(["cases.case_id", "diagnoses.diagnosis_id"]).ngroups)
print("Unique (case, treatment) pairs:",
      df.groupby(["cases.case_id", "treatments.treatment_id"]).ngroups)

Total rows:               5546
Unique case_id:           1098
Unique submitter_id:      1098

Distribution of rows per case:
1      68
2       5
3     326
4     131
5     165
6     177
7      90
8      50
9      26
10     14
11     13
12     10
13      2
14      6
15      3
16      2
17      3
19      2
20      1
21      1
22      1
33      1
40      1
Name: count, dtype: int64

Unique (case, diagnosis) pairs: 1343
Unique (case, treatment) pairs: 5537


**Interpretation.** Rows ≈ (case × treatment) pairs
(5,546 vs 5,537), so the row explosion is driven by treatments,
not by diagnoses. Diagnosis contributes ~245 extra rows
(1,343 pairs vs 1,098 patients).

## 3. Collapse to one row per patient

**Rule (documented in `data/README.md`):**
1. Keep `diagnoses.diagnosis_is_primary_disease == "true"` (case-insensitive).
2. Drop all `treatments.*` columns (row-explosion driver).
3. Sort by `diagnoses.days_to_diagnosis`, then `drop_duplicates(cases.case_id)`.
4. Rows with `'--` in the primary-disease flag are excluded (2 patients).

In [4]:
print("Raw values of primary-disease flag:")
print(df["diagnoses.diagnosis_is_primary_disease"].value_counts(dropna=False))

is_primary = (
    df["diagnoses.diagnosis_is_primary_disease"]
      .astype(str).str.lower() == "true"
)
prim = df[is_primary].copy()

print("\nAfter primary filter:")
print("  rows:", len(prim), "| unique cases:", prim["cases.case_id"].nunique())

invalid = df[df["diagnoses.diagnosis_is_primary_disease"] == "'--"]
print("\nRows with missing flag ('--):", len(invalid),
      "| unique cases:", invalid["cases.case_id"].nunique())

Raw values of primary-disease flag:
diagnoses.diagnosis_is_primary_disease
true     4946
false     597
'--         3
Name: count, dtype: int64

After primary filter:
  rows: 4946 | unique cases: 1096

Rows with missing flag ('--): 3 | unique cases: 2


In [5]:
tx_cols     = [c for c in prim.columns if c.startswith("treatments.")]
prim_no_tx  = prim.drop(columns=tx_cols).copy()
print("After dropping treatment columns:", prim_no_tx.shape)

prim_no_tx["diagnoses.days_to_diagnosis"] = pd.to_numeric(
    prim_no_tx["diagnoses.days_to_diagnosis"], errors="coerce"
)
prim_no_tx = prim_no_tx.sort_values(
    ["cases.case_id", "diagnoses.days_to_diagnosis"]
)
patient = prim_no_tx.drop_duplicates(subset="cases.case_id", keep="first").copy()

print("\nFinal patient-level table:")
print("  shape:", patient.shape)
print("  unique cases:", patient["cases.case_id"].nunique())
print("  is_unique per case:", patient["cases.case_id"].is_unique)

os.makedirs("../data/processed", exist_ok=True)
patient.to_csv("../data/processed/clinical_patient_level.csv", index=False)
print("Saved → data/processed/clinical_patient_level.csv")

After dropping treatment columns: (4946, 158)

Final patient-level table:
  shape: (1096, 158)
  unique cases: 1096
  is_unique per case: True
Saved → data/processed/clinical_patient_level.csv


**Result.** One row per patient, `cases.case_id` unique.
Saved for all downstream phases as
`data/processed/clinical_patient_level.csv`.

Next section: mapping RNA-seq sample barcodes → patients and
selecting one sample per patient for the merged analysis table.